# Demo: Single-Cell RNA-seq Analysis Workflow

This notebook demonstrates a complete scRNA-seq analysis workflow using the analysis template.

**Goals:**
1. Show how to use the local analysis package
2. Demonstrate which steps to run locally vs. on a GPU cluster (Euler)
3. Illustrate the local <-> remote sync workflow via Git/GitHub

**Legend:**
- **Local** - Fast, good for development, code editing
- **GPU/Euler** - Heavy compute, model fitting, GPU-accelerated

---

## Setup & Imports - Local

In [ ]:
import warnings

import matplotlib.pyplot as plt
import scanpy as sc

# Local analysis package - edit these functions in src/myanalysis/
from myanalysis import FilePaths, qc_violin, styled_umap

warnings.filterwarnings("ignore", category=FutureWarning)
sc.settings.verbosity = 2

print(f"Project root: {FilePaths.ROOT}")
print(f"Data folder:  {FilePaths.DATA}")

## Load Data - Local

We will use the classic PBMC 3k dataset from 10X Genomics, available via scanpy.

In [ ]:
# Download PBMC 3k (cached after first download)
adata = sc.datasets.pbmc3k()
adata

In [ ]:
# Store raw counts for scVI (needs raw integer counts)
adata.layers["counts"] = adata.X.copy()

## Quality Control - Local

QC is fast and benefits from quick iteration - perfect for local development.

In [ ]:
# Annotate mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")

# Calculate QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

In [ ]:
# Use our custom plotting function from the local package!
# TIP: Edit src/myanalysis/plotting.py locally, then re-import to see changes
fig = qc_violin(adata, figsize=(12, 3))
plt.show()

In [ ]:
# Filter cells and genes
print(f"Before filtering: {adata.n_obs} cells, {adata.n_vars} genes")

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.n_genes_by_counts < 2500, :]
adata = adata[adata.obs.pct_counts_mt < 5, :]

print(f"After filtering:  {adata.n_obs} cells, {adata.n_vars} genes")

## Preprocessing - Local

Normalization and HVG selection are fast operations.

In [ ]:
# Normalize and log-transform
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

# Store normalized counts
adata.raw = adata

# Identify highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor="seurat_v3", layer="counts")
print(f"Highly variable genes: {adata.var.highly_variable.sum()}")

---

## scVI Model Training - GPU/Euler

**This section benefits from GPU acceleration!**

Workflow for running on Euler:
1. Local: Commit and push your notebook: `git add . && git commit -m "Ready for scVI" && git push`
2. SSH to Euler, pull changes: `git pull`
3. Run this section in JupyterHub: https://jupyter.euler.hpc.ethz.ch
4. Save results and push: `git add . && git commit -m "scVI trained" && git push`
5. Local: Pull results: `git pull`

In [ ]:
import scvi
import torch

# Check if GPU is available
device = torch.cuda.get_device_name() if torch.cuda.is_available() else "CPU"
print(f"PyTorch device: {device}")
print(f"MPS available (Apple Silicon): {torch.backends.mps.is_available()}")

In [ ]:
# Setup scVI model
scvi.model.SCVI.setup_anndata(adata, layer="counts")

# Create and train model
# On GPU: ~1 min | On CPU: ~5-10 min
model = scvi.model.SCVI(adata, n_latent=10, n_layers=2)
model.train(max_epochs=100, early_stopping=True)

In [ ]:
# Get latent representation
adata.obsm["X_scVI"] = model.get_latent_representation()
print(f"scVI latent shape: {adata.obsm['X_scVI'].shape}")

---

## Neighbors & UMAP - GPU/Euler (rapids-singlecell)

On Linux with NVIDIA GPU, we can use `rapids-singlecell` for massive speedups.
Falls back to scanpy on macOS.

In [ ]:
import sys

# Use rapids-singlecell on Linux (GPU), scanpy on macOS
if sys.platform == "linux":
    try:
        import rapids_singlecell as rsc

        print("Using rapids-singlecell (GPU-accelerated)")
        USE_RAPIDS = True
    except ImportError:
        print("rapids-singlecell not available, falling back to scanpy")
        USE_RAPIDS = False
else:
    print("macOS detected, using scanpy (CPU)")
    USE_RAPIDS = False

In [ ]:
# Compute neighbors and UMAP
# With rapids on GPU: seconds | With scanpy on CPU: ~30s for 3k cells

if USE_RAPIDS:
    # GPU-accelerated
    rsc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)
    rsc.tl.umap(adata)
else:
    # CPU fallback
    sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)
    sc.tl.umap(adata)

In [ ]:
# Clustering (Leiden)
if USE_RAPIDS:
    rsc.tl.leiden(adata, resolution=0.5, key_added="leiden")
else:
    sc.tl.leiden(adata, resolution=0.5, key_added="leiden")

print(f"Found {adata.obs['leiden'].nunique()} clusters")

---

## Visualization - Local

Back to local for visualization and figure generation.

In [ ]:
# Use our custom styled_umap function
fig = styled_umap(adata, color="leiden", title="Leiden Clusters", figsize=(7, 6))
plt.show()

In [ ]:
# Marker genes
sc.pl.umap(adata, color=["CST3", "NKG7", "PPBP", "CD8A"], ncols=2, frameon=False)

## Cell Type Annotation - Local

In [ ]:
# Find marker genes per cluster
sc.tl.rank_genes_groups(adata, groupby="leiden", method="wilcoxon")
sc.pl.rank_genes_groups(adata, n_genes=5, sharey=False)

In [ ]:
# Manual annotation based on known markers
cluster_annotations = {
    "0": "CD4+ T cells",
    "1": "CD14+ Monocytes",
    "2": "B cells",
    "3": "CD8+ T cells",
    "4": "NK cells",
    "5": "FCGR3A+ Monocytes",
    "6": "Dendritic cells",
    "7": "Megakaryocytes",
}

adata.obs["cell_type"] = adata.obs["leiden"].map(cluster_annotations)
adata.obs["cell_type"].value_counts()

In [ ]:
# Final UMAP with cell types
fig = styled_umap(adata, color="cell_type", title="PBMC Cell Types", figsize=(8, 6))
plt.show()

---

## Save Results - Local

Save processed data following the template data organization.

In [ ]:
# Create output directory
output_dir = FilePaths.DATA / "pbmc3k" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

# Save processed AnnData
output_path = output_dir / "pbmc3k_analyzed.h5ad"
adata.write(output_path)
print(f"Saved to: {output_path}")

---

## Workflow Summary

| Step | Where | Why |
|------|-------|-----|
| QC & filtering | Local | Fast iteration, quick edits |
| Custom plotting | Local | Edit `src/myanalysis/plotting.py` |
| scVI training | GPU/Euler | 10-100x faster on GPU |
| Neighbors/UMAP | GPU/Euler | rapids-singlecell acceleration |
| Visualization | Local | Interactive exploration |

**Git sync workflow:**
```bash
# Local: push changes
git add . && git commit -m "description" && git push

# Euler: pull and run
git pull
# ... run heavy compute ...
git add . && git commit -m "results" && git push

# Local: pull results
git pull
```